# Chronological History and Feature Engineering

Loads the frozen Step 2 feature contract and one small real Parquet batch to demonstrate ordering, history fields, and leakage controls.

**Safety:** default execution is lightweight. Expensive full-data work runs only behind an explicit environment flag.

In [ ]:
from pathlib import Path
import os, json
import pandas as pd

candidate = Path(os.environ.get('ASSISTMENTS_WORKSPACE', Path.cwd().parents[1] / 'ASSISTments Dataset')).resolve()
if not candidate.exists():
    raise FileNotFoundError('Set ASSISTMENTS_WORKSPACE to the local ASSISTments Dataset folder')
WORKSPACE = candidate
print('Using workspace:', WORKSPACE)

In [ ]:
import pyarrow.parquet as pq
step2 = WORKSPACE / '02_Chronological_Learner_History'
spec = pd.read_csv(step2 / 'outputs' / 'ASSISTments_Step2_History_Feature_Specification.csv')
display(spec.groupby('history_configuration')['feature_name'].agg(['count', list]))
assert spec.groupby('history_configuration').size().to_dict() == {'H0':14, 'H1':15, 'H2':13}
print('Cumulative counts: H0=14, H1=29, H2=42')

In [ ]:
history_file = step2 / 'data' / 'processed' / 'ASSISTments_2012_2013_history_features.parquet'
columns = ['user_id','start_time_parsed','problem_log_id','correct_raw','target_eligible',
           'target_first_attempt_correct','prior_interaction_count','previous_correct_raw',
           'prior_correct_rate','recent3_correct_rate','flag_timestamp_tie']
batch = next(pq.ParquetFile(history_file).iter_batches(batch_size=500, columns=columns)).to_pandas()
display(batch.head(12))
ordered = batch.sort_values(['user_id','start_time_parsed','problem_log_id']).reset_index(drop=True)
assert batch.reset_index(drop=True).equals(ordered)
print('Sample is in frozen chronological order: PASS')

In [ ]:
tie_rows = batch[batch['flag_timestamp_tie'].fillna(False)]
print('Tie rows in this first batch:', len(tie_rows))
print('Pipeline rule: all peers sharing the current start timestamp are excluded from predictive history; problem_log_id is only a deterministic file-order tie-breaker.')
display(spec[['feature_name','history_configuration','mathematical_definition','timestamp_tie_behavior','leakage_risk']])